In [0]:
ruta_productos = "/Volumes/electrocasa_dev/bronze/landing/productos/catalogo_productos.json"

productos = (
    spark.read
        .option("multiLine", "true")
        .json(ruta_productos)
)

productos.printSchema()
display(productos.limit(10))

In [0]:
catalogo = dbutils.widgets.get("catalogo")

ruta = f"/Volumes/{catalogo}/bronze/landing/productos"


spark.sql(f"""
CREATE TABLE IF NOT EXISTS {catalogo}.bronze.productos
""")


spark.sql(f"""
COPY INTO {catalogo}.bronze.productos
FROM (
    SELECT
        *,
        current_timestamp() AS fec_ingesta,
        _metadata.file_name AS archivo_origen,
        concat(
            _metadata.file_name,
            '_',
            cast(_metadata.file_modification_time AS STRING)
        ) AS id_lote
    FROM '{ruta}'
)
FILEFORMAT = JSON
FORMAT_OPTIONS (
    'mergeSchema' = 'true',
    'multiLine' = 'true'
)
COPY_OPTIONS (
    'mergeSchema' = 'true'
)
""")

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT producto_id) AS productos_unicos,
    COUNT(DISTINCT archivo_origen) AS archivos_origen
FROM electrocasa_dev.bronze.productos;

In [0]:
%sql

SELECT
    producto_id,
    COUNT(*) AS cantidad
FROM electrocasa_dev.bronze.productos
GROUP BY producto_id
HAVING COUNT(*) > 1
ORDER BY cantidad DESC, producto_id;

In [0]:
%sql

SELECT
    producto_id,
    nombre_producto,
    categoria,
    marca,
    precio_lista,
    archivo_origen,
    id_lote
FROM electrocasa_dev.bronze.productos
WHERE producto_id IN (
    'P00097',
    'P00122',
    'P00130',
    'P00221',
    'P00227'
)
ORDER BY producto_id;

In [0]:
%sql

SELECT
    producto_id,
    COUNT(DISTINCT nombre_producto) AS nombres_distintos,
    COUNT(DISTINCT categoria) AS categorias_distintas,
    COUNT(DISTINCT marca) AS marcas_distintas,
    COUNT(DISTINCT precio_lista) AS precios_distintos
FROM electrocasa_dev.bronze.productos
WHERE producto_id IN (
    SELECT producto_id
    FROM electrocasa_dev.bronze.productos
    GROUP BY producto_id
    HAVING COUNT(*) > 1
)
GROUP BY producto_id
ORDER BY producto_id;

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,

    SUM(CASE
        WHEN precio_lista IS NULL OR TRIM(precio_lista) = ''
        THEN 1 ELSE 0
    END) AS precio_nulo,

    SUM(CASE
        WHEN precio_lista LIKE 'S/%'
        THEN 1 ELSE 0
    END) AS precio_con_moneda,

    SUM(CASE
        WHEN TRY_CAST(
            REGEXP_REPLACE(precio_lista, '^S/\\s*', '')
            AS DOUBLE
        ) <= 0
        THEN 1 ELSE 0
    END) AS precio_no_positivo

FROM electrocasa_dev.bronze.productos;

In [0]:
%sql

SELECT
    SUM(CASE
        WHEN marca IS NULL OR TRIM(marca) = ''
        THEN 1 ELSE 0
    END) AS marca_faltante,

    COUNT(DISTINCT categoria) AS categorias_distintas

FROM electrocasa_dev.bronze.productos;

In [0]:
%sql

SELECT
    categoria,
    COUNT(*) AS cantidad
FROM electrocasa_dev.bronze.productos
GROUP BY categoria
ORDER BY categoria;

In [0]:
%sql

SELECT
    COUNT(*) AS total_registros,
    COUNT(DISTINCT producto_id) AS productos_unicos,
    COUNT(DISTINCT categoria) AS categorias_distintas,

    SUM(CASE
        WHEN marca IS NULL OR TRIM(marca) = ''
        THEN 1 ELSE 0
    END) AS marca_faltante,

    SUM(CASE
        WHEN precio_lista IS NULL
        THEN 1 ELSE 0
    END) AS precio_nulo

FROM electrocasa_dev.silver.productos;

In [0]:
%sql

SELECT
    categoria,
    COUNT(*) AS cantidad
FROM electrocasa_dev.silver.productos
GROUP BY categoria
ORDER BY categoria;

In [0]:
%sql

DESCRIBE TABLE electrocasa_dev.silver.productos;